# Amazon Bedrock Streaming API: `InvokeModelWithResponseStream` & `ConverseStream`

*This notebook provides an in-depth, hands-on guide to real-time response streaming using **Amazon first-party models** (Amazon Nova and Amazon Titan). Adapted and expanded from [AWS Samples: 01_invoke_api.ipynb](https://github.com/aws-samples/amazon-bedrock-samples/blob/main/introduction-to-bedrock/bedrock_apis/01_invoke_api.ipynb).*

---

## 1. Why Streaming Matters

When invoking Large Language Models (LLMs) synchronously, the client must wait until the model finishes generating the entire completion before receiving any output. For long answers, this latency can range from 3 to 15+ seconds.

### Benefits of Response Streaming:
1. **Dramatic Reduction in Perceived Latency**: Output begins rendering on the user's screen within **200–500 milliseconds** (Time to First Token), rather than waiting for the entire response to complete.
2. **Real-Time Interactive Feedback**: Essential for conversational chatbots, streaming web UIs, and real-time coding assistants.
3. **Early Cancellation / Abort**: Users can cancel or stop generation early if the model has already provided the desired answer, saving downstream tokens.
4. **Identical Cost**: Amazon Bedrock charges the exact same per-token rate for streaming as for synchronous calls—there is **no price premium** for streaming.

### How It Works Under the Hood: The AWS EventStream Protocol
Amazon Bedrock streams responses over **HTTP/2** using the binary `application/vnd.amazon.eventstream` format. The server yields binary events over an open HTTP socket. Each event contains headers and a payload chunk that is decoded into JSON.


## 2. Setup & Client Initialization

We initialize the `bedrock-runtime` client in our target region (`us-east-2` or `us-east-1`):


In [ ]:
import boto3
import json
import sys
from datetime import datetime
from IPython.display import clear_output, display, Markdown

# Configure region
REGION = "us-east-2"
bedrock_runtime = boto3.client(service_name="bedrock-runtime", region_name=REGION)

print(f"Bedrock Runtime client successfully initialized for region: {REGION}")


## 3. Streaming with Amazon Nova Models (`invoke_model_with_response_stream`)

Amazon Nova models (`us.amazon.nova-lite-v1:0`, `us.amazon.nova-micro-v1:0`, `us.amazon.nova-pro-v1:0`) use the unified `messages-v1` schema.

During streaming, Bedrock yields chunks. Within each chunk payload, text tokens arrive in the `contentBlockDelta` structure.

Let's stream a response while measuring **Time to First Token (TTFT)**, **Total Chunks**, and **Streaming Speed (Tokens/Second)**:


In [ ]:
prompt_text = "Describe three traditional dance styles from different cultures around the world. Explain their origins and cultural significance."

nova_payload = {
    "schemaVersion": "messages-v1",
    "messages": [
        {"role": "user", "content": [{"text": prompt_text}]}
    ],
    "inferenceConfig": {
        "max_new_tokens": 450,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 20
    }
}

NOVA_MODEL_ID = "us.amazon.nova-lite-v1:0"  # or amazon.nova-lite-v1:0

start_time = datetime.now()
time_to_first_token = None
chunk_count = 0
full_response = ""

# Initiate the streaming call
response = bedrock_runtime.invoke_model_with_response_stream(
    modelId=NOVA_MODEL_ID,
    body=json.dumps(nova_payload),
    contentType="application/json",
    accept="application/json"
)

print(f"--- Streaming from {NOVA_MODEL_ID} ---\n")
stream = response.get("body")

if stream:
    for event in stream:
        chunk = event.get("chunk")
        if chunk:
            chunk_json = json.loads(chunk.get("bytes").decode("utf-8"))
            content_block_delta = chunk_json.get("contentBlockDelta")
            if content_block_delta:
                if time_to_first_token is None:
                    time_to_first_token = datetime.now() - start_time
                
                chunk_count += 1
                text_piece = content_block_delta.get("delta", {}).get("text", "")
                full_response += text_piece
                print(text_piece, end="", flush=True)

    total_time = datetime.now() - start_time
    print(f"\n\n{'=' * 50}")
    print(f"Time to First Token (TTFT): {time_to_first_token.total_seconds():.3f} seconds")
    print(f"Total Stream Elapsed Time:   {total_time.total_seconds():.3f} seconds")
    print(f"Total Chunks Received:       {chunk_count}")


## 4. Deep Dive: Anatomy of the Amazon Nova Stream Lifecycle

What exactly happens under the hood during an Amazon Nova stream? An Amazon Nova stream emits a deterministic sequence of events from start to finish:

1. **`messageStart`**: Emitted once at the beginning; contains the role (`assistant`).
2. **`contentBlockStart`**: Emitted when a content block begins (contains `contentBlockIndex: 0`).
3. **`contentBlockDelta`**: Emitted repeatedly for each increment of generated text tokens.
4. **`contentBlockStop`**: Emitted when the content block generation ends.
5. **`messageStop`**: Emitted at the very end; contains `stopReason` (`end_turn`, `max_tokens`) and **`amazon-bedrock-invocationMetrics`** (input tokens, output tokens, invocation latency, first-byte latency).

Let's inspect every raw event in the stream to see this lifecycle in action:


In [ ]:
test_payload = {
    "schemaVersion": "messages-v1",
    "messages": [{"role": "user", "content": [{"text": "Give a one-sentence inspiring quote about engineering."}]}],
    "inferenceConfig": {"max_new_tokens": 60, "temperature": 0.3}
}

response = bedrock_runtime.invoke_model_with_response_stream(
    modelId="us.amazon.nova-lite-v1:0",
    body=json.dumps(test_payload),
    contentType="application/json",
    accept="application/json"
)

event_types_seen = []
final_metrics = {}

for event in response.get("body"):
    chunk = event.get("chunk")
    if chunk:
        data = json.loads(chunk.get("bytes").decode("utf-8"))
        for key in data.keys():
            if key not in event_types_seen:
                event_types_seen.append(key)
        
        # If this is the messageStop event, capture metrics
        if "messageStop" in data:
            final_metrics = data["messageStop"]

print("Observed Event Types in Nova Stream Lifecycle:")
for i, et in enumerate(event_types_seen, 1):
    print(f"  {i}. {et}")

print("\n--- Final Stream Metrics (from messageStop) ---")
print(json.dumps(final_metrics, indent=2))


## 5. Modern Unified Streaming with the `converse_stream` API

Amazon Bedrock provides the **`converse_stream`** API as the unified, cross-model streaming interface.

### Why Use `converse_stream`?
- **Normalized Input**: You don't need model-specific request JSON bodies (no need to remember Nova's `messages-v1` vs Titan's `inputText`).
- **Normalized Output**: Events (`messageStart`, `contentBlockDelta`, `messageStop`, `metadata`) are standardized across all foundation models.
- **Built-in System Prompts & Guardrails**: Easily attach `system=[...]` and `guardrailConfig={...}`.


In [ ]:
# Stream using Bedrock's unified converse_stream API
response = bedrock_runtime.converse_stream(
    modelId="us.amazon.nova-lite-v1:0",
    messages=[
        {
            "role": "user",
            "content": [{"text": "Explain what an API gateway is and why it is useful in microservices."}]
        }
    ],
    system=[
        {"text": "You are an expert cloud solutions architect. Provide clear, concise explanations."}
    ],
    inferenceConfig={
        "maxTokens": 350,
        "temperature": 0.5
    }
)

print("--- Streaming via converse_stream ---\n")

usage_metadata = {}
stop_reason = ""

for event in response.get("stream"):
    # Text chunk delta
    if "contentBlockDelta" in event:
        text = event["contentBlockDelta"].get("delta", {}).get("text", "")
        print(text, end="", flush=True)
    
    # End of message stop reason
    elif "messageStop" in event:
        stop_reason = event["messageStop"].get("stopReason", "")
    
    # Final token usage and latency metrics
    elif "metadata" in event:
        usage_metadata = event["metadata"]

print(f"\n\n{'=' * 50}")
print(f"Stop Reason: {stop_reason}")
print("Usage & Latency Metadata:", json.dumps(usage_metadata, indent=2))


## 6. Streaming with Amazon Titan Text Models

Amazon Titan Text models (`amazon.titan-text-express-v1`) yield streaming chunks with an `outputText` field:


In [ ]:
titan_payload = {
    "inputText": "Give 3 best practices for designing RESTful APIs.",
    "textGenerationConfig": {
        "maxTokenCount": 300,
        "temperature": 0.5,
        "topP": 0.9
    }
}

TITAN_MODEL_ID = "amazon.titan-text-express-v1"

try:
    response = bedrock_runtime.invoke_model_with_response_stream(
        modelId=TITAN_MODEL_ID,
        body=json.dumps(titan_payload),
        contentType="application/json",
        accept="application/json"
    )
    
    print(f"--- Streaming from {TITAN_MODEL_ID} ---\n")
    for event in response.get("body"):
        chunk = event.get("chunk")
        if chunk:
            chunk_json = json.loads(chunk.get("bytes").decode("utf-8"))
            output_text = chunk_json.get("outputText", "")
            print(output_text, end="", flush=True)
    print()
except Exception as e:
    print(f"Note: Verify {TITAN_MODEL_ID} access is enabled in Bedrock: {e}")


## 7. Interactive Live Markdown Rendering

In interactive notebooks, we can re-render Markdown dynamically as chunks arrive using `IPython.display` (`clear_output`, `display`, `Markdown`):


In [ ]:
prompt_text = "Create a formatted Markdown table comparing Python, JavaScript, and Go on syntax, typing, and concurrency."

nova_payload = {
    "schemaVersion": "messages-v1",
    "messages": [{"role": "user", "content": [{"text": prompt_text}]}],
    "inferenceConfig": {"max_new_tokens": 350, "temperature": 0.3}
}

response = bedrock_runtime.invoke_model_with_response_stream(
    modelId="us.amazon.nova-lite-v1:0",
    body=json.dumps(nova_payload),
    contentType="application/json",
    accept="application/json"
)

accumulated_text = ""
update_interval = 4  # Re-render every 4 chunks to balance smoothness and render cost
counter = 0

for event in response.get("body"):
    chunk = event.get("chunk")
    if chunk:
        chunk_json = json.loads(chunk.get("bytes").decode("utf-8"))
        delta = chunk_json.get("contentBlockDelta", {}).get("delta", {}).get("text", "")
        accumulated_text += delta
        counter += 1
        
        if counter % update_interval == 0:
            clear_output(wait=True)
            display(Markdown(accumulated_text))

# Final render to ensure completion is completely displayed
clear_output(wait=True)
display(Markdown(accumulated_text))


## 8. Production Pattern: Building a Reusable Python Streaming Generator

When building web applications (FastAPI `StreamingResponse`, Flask SSE, or Streamlit `st.write_stream`), wrapping Bedrock streaming in a clean Python generator is the standard production pattern:


In [ ]:
from typing import Iterator

def stream_nova_generator(prompt: str, model_id: str = "us.amazon.nova-lite-v1:0") -> Iterator[str]:
    """
    Reusable generator function that yields text tokens from Amazon Nova as they arrive.
    Can be plugged directly into web streaming endpoints (FastAPI / Streamlit).
    """
    payload = {
        "schemaVersion": "messages-v1",
        "messages": [{"role": "user", "content": [{"text": prompt}]}],
        "inferenceConfig": {"max_new_tokens": 300, "temperature": 0.5}
    }
    
    try:
        response = bedrock_runtime.invoke_model_with_response_stream(
            modelId=model_id,
            body=json.dumps(payload),
            contentType="application/json",
            accept="application/json"
        )
        
        stream = response.get("body")
        if stream:
            for event in stream:
                chunk = event.get("chunk")
                if chunk:
                    data = json.loads(chunk["bytes"].decode("utf-8"))
                    delta = data.get("contentBlockDelta", {}).get("delta", {}).get("text", "")
                    if delta:
                        yield delta
    except Exception as e:
        yield f"\n[Stream Error: {str(e)}]"

# Example consumption of the generator:
print("Testing generator output:\n")
for token in stream_nova_generator("List 3 essential leadership principles in bullet points."):
    print(token, end="", flush=True)


## 9. Clean Up & Stream Lifecycle

### Stream Lifecycle Management
- **Automatic Socket Closure**: Bedrock HTTP/2 EventStreams automatically close when generation completes (when an end-of-turn or max token event is received).
- **Avoid Socket Leaks**: Always iterate through the stream to completion or wrap it in a `try...finally` block so underlying HTTP connections return cleanly to the connection pool.
- **Mid-Stream Error Handling**: In production, wrap stream iteration in exception handlers (`ModelStreamErrorException`, `ThrottlingException`) to gracefully notify clients if a network dropout occurs mid-stream.
- **Pricing & Billing**: Amazon Bedrock charges the exact same per-token rate for streaming as for synchronous `invoke_model`. There are no additional or hidden fees for streaming.

### Recommended Cleanup Steps:
1. **Kernel Shutdown**: Shut down the notebook kernel (`Kernel` -> `Shut Down Kernel`) when done to terminate active worker threads and release local memory.
2. **Monitor Quotas**: Keep track of concurrent model invocation limits and quota utilization in AWS Service Quotas.

---
### Related Notebooks
- [001-Invoke-model.ipynb](001-Invoke-model.ipynb): Learn non-streaming invocation, inference parameters, and image generation with Amazon models.
- [003-Guardrails.ipynb](003-Guardrails.ipynb): Learn how to attach safety guardrails to streaming and non-streaming models.
